# 05 — Late Fusion Baseline

Weighted late fusion of HuBERT audio predictions and TF-IDF + Logistic Regression text predictions on IEMOCAP (5-class: angry, happy, neutral, sad, frustrated).

**What this notebook does (no retraining involved):**
1. Loads text predictions from cache if they exist; otherwise generates them on the shared train/val/test split (fast TF-IDF + LR — seconds)
2. Loads the HuBERT val/test prediction CSVs produced by notebook 04
3. Runs sanity checks to confirm perfect alignment
4. Searches 21 audio/text weight combinations (0.00 → 1.00, step 0.05) on the **validation set only**
5. Evaluates the best weight on the **test set** and saves all artefacts

**Prerequisites:** notebooks 03 and 04 must have been run so that the following files exist:
- `outputs/whisper_cleaned_dataset.csv`
- `outputs/hubert_audio_baseline/shared_split_indices.csv`
- `outputs/hubert_audio_baseline/hubert_val_predictions_with_probs.csv`
- `outputs/hubert_audio_baseline/hubert_test_predictions_with_probs.csv`
- `outputs/hubert_audio_baseline/hubert_final_metrics.json`

---
## 1. Configuration

In [ ]:
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Directory layout
NOTEBOOK_DIR = Path('.')
OUTPUTS_DIR  = NOTEBOOK_DIR / 'outputs'
HUBERT_DIR   = OUTPUTS_DIR / 'hubert_audio_baseline'
FUSION_DIR   = OUTPUTS_DIR / 'fusion_baseline'
FUSION_DIR.mkdir(parents=True, exist_ok=True)

# Emotion classes — must match HuBERT label order
EMOTIONS   = ['angry', 'happy', 'neutral', 'sad', 'frustrated']
EMOTION2ID = {e: i for i, e in enumerate(EMOTIONS)}
PROB_COLS  = [f'prob_{e}' for e in EMOTIONS]

# Fusion weight grid: audio weight from 0.0 to 1.0 inclusive, step 0.05 (21 points)
# text weight = 1 - audio weight
WEIGHT_GRID = np.round(np.arange(0.0, 1.01, 0.05), 2).tolist()

print(f'Output directory : {FUSION_DIR.resolve()}')
print(f'Emotion classes  : {EMOTIONS}')
print(f'Weight grid ({len(WEIGHT_GRID)} points): {WEIGHT_GRID}')

---
## 2. Generate Aligned Text Predictions

If `text_val_predictions_with_probs.csv` and `text_test_predictions_with_probs.csv` already exist in the fusion output directory they are loaded directly — no model fitting occurs.

Otherwise, the same TF-IDF + Logistic Regression pipeline from notebook 03 is fitted on the **shared train split** (from `shared_split_indices.csv`) to produce prediction files in the format required for fusion. This takes ~10 seconds.

In [ ]:
TEXT_VAL_PATH  = FUSION_DIR / 'text_val_predictions_with_probs.csv'
TEXT_TEST_PATH = FUSION_DIR / 'text_test_predictions_with_probs.csv'

if TEXT_VAL_PATH.exists() and TEXT_TEST_PATH.exists():
    # ── Cache hit: load existing prediction files ─────────────────────────
    text_val_full_df  = pd.read_csv(TEXT_VAL_PATH)
    text_test_full_df = pd.read_csv(TEXT_TEST_PATH)
    print('Text predictions loaded from cache.')
    print(f'  {TEXT_VAL_PATH.name}  ({len(text_val_full_df):,} rows)')
    print(f'  {TEXT_TEST_PATH.name} ({len(text_test_full_df):,} rows)')

else:
    # ── Cache miss: fit TF-IDF + LR on the shared train split ────────────
    print('Text prediction files not found — fitting TF-IDF + LR ...')

    whisper_df = pd.read_csv(OUTPUTS_DIR / 'whisper_cleaned_dataset.csv')
    split_df   = pd.read_csv(HUBERT_DIR  / 'shared_split_indices.csv')

    # Join text onto split index via filename
    merged_df = split_df.merge(
        whisper_df[['file', 'cleaned_text']], on='file', how='left'
    )
    merged_df['cleaned_text'] = merged_df['cleaned_text'].fillna('')

    train_rows = merged_df[merged_df['split'] == 'train'].reset_index(drop=True)
    val_rows   = merged_df[merged_df['split'] == 'val'  ].reset_index(drop=True)
    test_rows  = merged_df[merged_df['split'] == 'test' ].reset_index(drop=True)

    print(f'  Train : {len(train_rows):,}   Val : {len(val_rows):,}   Test : {len(test_rows):,}')

    # TF-IDF vectorizer — same hyperparameters as notebook 03
    tfidf = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2),
        stop_words='english', sublinear_tf=True, min_df=2,
    )
    X_train_tfidf = tfidf.fit_transform(train_rows['cleaned_text'].values)
    X_val_tfidf   = tfidf.transform(val_rows['cleaned_text'].values)
    X_test_tfidf  = tfidf.transform(test_rows['cleaned_text'].values)

    # Logistic Regression — same hyperparameters as notebook 03
    lr = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_SEED)
    lr.fit(X_train_tfidf, train_rows['emotion'].values)

    print(f'  Vocabulary : {len(tfidf.get_feature_names_out()):,} features')
    print(f'  Classes    : {lr.classes_.tolist()}')

    def _build_text_pred_df(X_tfidf, split_rows_df):
        """Produce a prediction DataFrame with columns aligned to EMOTIONS."""
        proba       = lr.predict_proba(X_tfidf)
        class_order = lr.classes_.tolist()
        prob_matrix = np.zeros((len(proba), len(EMOTIONS)))
        for out_col, emotion in enumerate(EMOTIONS):
            if emotion in class_order:
                prob_matrix[:, out_col] = proba[:, class_order.index(emotion)]
        pred_labels = [EMOTIONS[i] for i in np.argmax(prob_matrix, axis=1)]
        out = split_rows_df[['orig_idx', 'file', 'emotion']].copy()
        out = out.rename(columns={'emotion': 'true_label'})
        out['pred_label'] = pred_labels
        for i, col in enumerate(PROB_COLS):
            out[col] = prob_matrix[:, i]
        return out.sort_values('orig_idx').reset_index(drop=True)

    text_val_full_df  = _build_text_pred_df(X_val_tfidf,  val_rows)
    text_test_full_df = _build_text_pred_df(X_test_tfidf, test_rows)

    text_val_full_df.to_csv( TEXT_VAL_PATH,  index=False)
    text_test_full_df.to_csv(TEXT_TEST_PATH, index=False)
    print('  Text prediction files generated and saved.')

# Report metrics regardless of whether files were loaded or generated
print()
for split_name, df in [('val', text_val_full_df), ('test', text_test_full_df)]:
    uar = balanced_accuracy_score(df['true_label'], df['pred_label'])
    acc = accuracy_score(df['true_label'], df['pred_label'])
    f1  = f1_score(df['true_label'], df['pred_label'], average='macro', zero_division=0)
    print(f'Text {split_name:4s} — Acc: {acc:.4f}  UAR: {uar:.4f}  Macro F1: {f1:.4f}  ({len(df):,} samples)')

---
## 3. Load All Predictions

In [ ]:
# HuBERT predictions (from notebook 04)
hubert_val_raw  = pd.read_csv(HUBERT_DIR / 'hubert_val_predictions_with_probs.csv')
hubert_test_raw = pd.read_csv(HUBERT_DIR / 'hubert_test_predictions_with_probs.csv')

# Standardise column names: HuBERT uses true_emotion / pred_emotion
for df in (hubert_val_raw, hubert_test_raw):
    df.rename(columns={'true_emotion': 'true_label', 'pred_emotion': 'pred_label'}, inplace=True)

# Text predictions (loaded or generated in section 2)
text_val_raw  = pd.read_csv(TEXT_VAL_PATH)
text_test_raw = pd.read_csv(TEXT_TEST_PATH)

print('Raw prediction counts:')
print(f'  HuBERT val  : {len(hubert_val_raw):>5,} samples')
print(f'  HuBERT test : {len(hubert_test_raw):>5,} samples')
print(f'  Text   val  : {len(text_val_raw):>5,} samples')
print(f'  Text   test : {len(text_test_raw):>5,} samples')
print()
if len(hubert_val_raw) != len(text_val_raw):
    print('Note: sample counts differ — HuBERT was likely run in DEBUG mode.')
    print('Aligning on the intersection of orig_idx values...')

In [ ]:
def align_on_orig_idx(audio_df, text_df, split_name):
    """Inner join on orig_idx; sort both by orig_idx for identical ordering."""
    audio_sorted = audio_df.sort_values('orig_idx').reset_index(drop=True)
    text_sorted  = text_df.sort_values('orig_idx').reset_index(drop=True)

    common_idx = set(audio_sorted['orig_idx']) & set(text_sorted['orig_idx'])

    audio_aligned = audio_sorted[audio_sorted['orig_idx'].isin(common_idx)].reset_index(drop=True)
    text_aligned  = text_sorted[ text_sorted['orig_idx'].isin(common_idx)].reset_index(drop=True)

    dropped_audio = len(audio_sorted) - len(audio_aligned)
    dropped_text  = len(text_sorted)  - len(text_aligned)
    if dropped_audio:
        print(f'[{split_name}] Dropped {dropped_audio} HuBERT rows not found in text set')
    if dropped_text:
        print(f'[{split_name}] Dropped {dropped_text} text rows not found in HuBERT set')

    return audio_aligned, text_aligned


audio_val,  text_val  = align_on_orig_idx(hubert_val_raw,  text_val_raw,  'val')
audio_test, text_test = align_on_orig_idx(hubert_test_raw, text_test_raw, 'test')

print(f'Aligned val  : {len(audio_val):,} samples')
print(f'Aligned test : {len(audio_test):,} samples')

---
## 4. Sanity Checks

Before any fusion, verify that the aligned prediction DataFrames are fully compatible.

In [ ]:
def run_sanity_checks(audio_df, text_df, split_name, prob_tol=1e-4):
    """Raise ValueError if any alignment or integrity check fails."""
    errors = []

    # 1. Equal sample counts
    if len(audio_df) != len(text_df):
        errors.append(f'Sample count mismatch — audio: {len(audio_df)}, text: {len(text_df)}')

    # 2. Identical orig_idx values
    if not np.array_equal(audio_df['orig_idx'].values, text_df['orig_idx'].values):
        errors.append('orig_idx values differ between audio and text DataFrames')

    # 3. Identical ordering (element-wise)
    if audio_df['orig_idx'].tolist() != text_df['orig_idx'].tolist():
        errors.append('orig_idx ordering differs between audio and text DataFrames')

    # 4. Identical true labels
    if not np.array_equal(audio_df['true_label'].values, text_df['true_label'].values):
        n_diff = (audio_df['true_label'].values != text_df['true_label'].values).sum()
        errors.append(f'true_label differs in {n_diff} rows')

    # 5. Probability rows sum to 1
    for tag, df in [('audio', audio_df), ('text', text_df)]:
        prob_sums = df[PROB_COLS].sum(axis=1)
        max_dev   = (prob_sums - 1.0).abs().max()
        if max_dev > prob_tol:
            errors.append(f'{tag} probabilities do not sum to 1 (max deviation = {max_dev:.6f})')

    if errors:
        for msg in errors:
            print(f'  [FAIL] {msg}')
        raise ValueError(f'Sanity checks FAILED for {split_name} ({len(errors)} error(s)). '
                         'Fix the issues above before running fusion.')

    print(f'[{split_name}] All 5 sanity checks PASSED  ({len(audio_df):,} aligned samples)')


run_sanity_checks(audio_val,  text_val,  'val')
run_sanity_checks(audio_test, text_test, 'test')

---
## 5. Validation Fusion Search

Sweep over 21 audio/text weight combinations (audio weight 0.00 → 1.00, step 0.05).  
Model selection is based **exclusively on validation UAR** — test results are not consulted.

In [ ]:
def weighted_fusion(audio_df, text_df, audio_weight):
    """Weighted average of probability vectors; predict via argmax."""
    text_weight  = 1.0 - audio_weight
    audio_probs  = audio_df[PROB_COLS].values
    text_probs   = text_df[PROB_COLS].values

    fusion_probs = audio_weight * audio_probs + text_weight * text_probs
    pred_labels  = [EMOTIONS[i] for i in np.argmax(fusion_probs, axis=1)]
    true_labels  = audio_df['true_label'].tolist()

    acc = accuracy_score(true_labels, pred_labels)
    f1  = f1_score(true_labels, pred_labels, average='macro', zero_division=0)
    uar = balanced_accuracy_score(true_labels, pred_labels)

    return pred_labels, fusion_probs, acc, f1, uar

In [ ]:
val_records = []
for audio_w in WEIGHT_GRID:
    _, _, acc, f1, uar = weighted_fusion(audio_val, text_val, audio_w)
    val_records.append({
        'audio_weight': audio_w,
        'text_weight':  round(1.0 - audio_w, 2),
        'val_accuracy': round(acc, 4),
        'val_macro_f1': round(f1,  4),
        'val_uar':      round(uar, 4),
    })

val_results_df = pd.DataFrame(val_records)

print('Validation Fusion Search Results:')
print(val_results_df.to_string(index=False))

---
## 6. Best Weight Selection

Select the weight with the highest **validation UAR**.  
Test results are not consulted at this stage.

In [ ]:
best_row     = val_results_df.loc[val_results_df['val_uar'].idxmax()]
BEST_AUDIO_W = float(best_row['audio_weight'])
BEST_TEXT_W  = float(best_row['text_weight'])
BEST_VAL_UAR = float(best_row['val_uar'])
BEST_VAL_F1  = float(best_row['val_macro_f1'])
BEST_VAL_ACC = float(best_row['val_accuracy'])

print(f'Best audio weight   : {BEST_AUDIO_W}')
print(f'Best text  weight   : {BEST_TEXT_W}')
print(f'Validation UAR      : {BEST_VAL_UAR:.4f}')
print(f'Validation Macro F1 : {BEST_VAL_F1:.4f}')
print(f'Validation Accuracy : {BEST_VAL_ACC:.4f}')

---
## 7. Test Evaluation

Evaluate the best-weight fusion model on the held-out **test set**.

In [ ]:
test_pred_labels, test_fusion_probs, test_acc, test_f1, test_uar = \
    weighted_fusion(audio_test, text_test, BEST_AUDIO_W)

print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test UAR      : {test_uar:.4f}')
print(f'Test Macro F1 : {test_f1:.4f}')
print()
print('Classification Report:')
print(classification_report(
    audio_test['true_label'],
    test_pred_labels,
    target_names=EMOTIONS,
    zero_division=0,
))

In [ ]:
cm = confusion_matrix(
    audio_test['true_label'],
    test_pred_labels,
    labels=EMOTIONS,
)

cm_df = pd.DataFrame(cm, index=EMOTIONS, columns=EMOTIONS)
cm_df.index.name   = 'true'
cm_df.columns.name = 'pred'
print('Confusion Matrix (rows = true, cols = predicted):')
print(cm_df)

---
## 8. Visualizations

In [ ]:
matplotlib.rcParams.update({'font.size': 11})

x   = val_results_df['audio_weight'].values
# Major ticks every 0.1 so the axis is readable with 21 data points
major_ticks = np.round(np.arange(0.0, 1.01, 0.1), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Validation Fusion Weight Search  (21 points, step 0.05)', fontsize=13, fontweight='bold')

# --- UAR vs audio weight ---
ax = axes[0]
ax.plot(x, val_results_df['val_uar'], 'o-', color='steelblue', linewidth=2, markersize=5)
ax.axvline(BEST_AUDIO_W, color='crimson', linestyle='--', linewidth=1.5, alpha=0.85,
           label=f'Best audio = {BEST_AUDIO_W}  (UAR {BEST_VAL_UAR:.4f})')
ax.set_title('Validation UAR vs Audio Weight')
ax.set_xlabel('Audio Weight  (text weight = 1 − audio)')
ax.set_ylabel('UAR (Balanced Accuracy)')
ax.set_xticks(major_ticks)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
ax.grid(axis='both', alpha=0.3)

# --- Macro F1 vs audio weight ---
ax = axes[1]
ax.plot(x, val_results_df['val_macro_f1'], 's-', color='darkorange', linewidth=2, markersize=5)
ax.axvline(BEST_AUDIO_W, color='crimson', linestyle='--', linewidth=1.5, alpha=0.85,
           label=f'Best audio = {BEST_AUDIO_W}  (F1 {BEST_VAL_F1:.4f})')
ax.set_title('Validation Macro F1 vs Audio Weight')
ax.set_xlabel('Audio Weight  (text weight = 1 − audio)')
ax.set_ylabel('Macro F1')
ax.set_xticks(major_ticks)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
ax.grid(axis='both', alpha=0.3)

plt.tight_layout()
plt.savefig(FUSION_DIR / 'fusion_weight_search.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fusion_weight_search.png')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=EMOTIONS)
disp.plot(ax=ax, cmap='Blues', colorbar=True)

ax.set_title(
    f'Fusion — Test Confusion Matrix\n'
    f'Audio weight = {BEST_AUDIO_W}  |  Text weight = {BEST_TEXT_W}  |  '
    f'UAR = {test_uar:.4f}',
    fontsize=12,
    fontweight='bold',
)
plt.tight_layout()
plt.savefig(FUSION_DIR / 'fusion_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fusion_confusion_matrix.png')

---
## 9. Final Comparison

Compare HuBERT, Text, and Fusion on the same test samples.

In [ ]:
# HuBERT test metrics — read from the saved JSON
with open(HUBERT_DIR / 'hubert_final_metrics.json') as f:
    hubert_metrics = json.load(f)

hubert_test_acc = hubert_metrics['final_test_accuracy']
hubert_test_uar = hubert_metrics['final_test_uar']
hubert_test_f1  = hubert_metrics['final_test_macro_f1']

# Text metrics on the aligned test samples (same set as HuBERT / fusion)
true_labels_test   = audio_test['true_label'].tolist()
text_preds_aligned = text_test['pred_label'].tolist()

text_test_acc = accuracy_score(true_labels_test, text_preds_aligned)
text_test_uar = balanced_accuracy_score(true_labels_test, text_preds_aligned)
text_test_f1  = f1_score(true_labels_test, text_preds_aligned, average='macro', zero_division=0)

comparison_df = pd.DataFrame([
    {'Model': 'HuBERT (audio)',    'Accuracy': hubert_test_acc, 'UAR': hubert_test_uar, 'Macro F1': hubert_test_f1},
    {'Model': 'Text (TF-IDF+LR)', 'Accuracy': text_test_acc,   'UAR': text_test_uar,   'Macro F1': text_test_f1},
    {'Model': 'Fusion (best)',     'Accuracy': test_acc,        'UAR': test_uar,        'Macro F1': test_f1},
]).set_index('Model').round(4)

print('Final Model Comparison — Test Set:')
print(comparison_df.to_string())

---
## 10. Fusion Improvement Analysis

In [ ]:
fusion_vs_hubert_uar = test_uar - hubert_test_uar
fusion_vs_text_uar   = test_uar - text_test_uar
fusion_vs_hubert_f1  = test_f1  - hubert_test_f1
fusion_vs_text_f1    = test_f1  - text_test_f1

print('Fusion improvement over HuBERT:')
print(f'  UAR      : {fusion_vs_hubert_uar:+.4f}')
print(f'  Macro F1 : {fusion_vs_hubert_f1:+.4f}')
print()
print('Fusion improvement over Text:')
print(f'  UAR      : {fusion_vs_text_uar:+.4f}')
print(f'  Macro F1 : {fusion_vs_text_f1:+.4f}')

---
## 11. Final Summary

In [ ]:
print('=' * 58)
print('FUSION BASELINE — FINAL SUMMARY')
print('=' * 58)
print(f'Best Fusion Weight  : Audio = {BEST_AUDIO_W}  |  Text = {BEST_TEXT_W}')
print(f'Validation UAR      : {BEST_VAL_UAR:.4f}')
print()
print(f'Test Accuracy       : {test_acc:.4f}')
print(f'Test UAR            : {test_uar:.4f}')
print(f'Test Macro F1       : {test_f1:.4f}')
print()
print('Improvement over HuBERT:')
print(f'  UAR      : {fusion_vs_hubert_uar:+.4f}')
print(f'  Macro F1 : {fusion_vs_hubert_f1:+.4f}')
print()
print('Improvement over Text:')
print(f'  UAR      : {fusion_vs_text_uar:+.4f}')
print(f'  Macro F1 : {fusion_vs_text_f1:+.4f}')
print('=' * 58)

---
## 12. Save Outputs

In [ ]:
# fusion_validation_results.csv
val_results_df.to_csv(FUSION_DIR / 'fusion_validation_results.csv', index=False)

# fusion_test_predictions.csv
test_preds_df = audio_test[['orig_idx', 'true_label']].copy()
test_preds_df['pred_label'] = test_pred_labels
for i, col in enumerate(PROB_COLS):
    test_preds_df[col] = test_fusion_probs[:, i]
test_preds_df.to_csv(FUSION_DIR / 'fusion_test_predictions.csv', index=False)

# fusion_final_metrics.json
final_metrics = {
    'best_audio_weight':    BEST_AUDIO_W,
    'best_text_weight':     BEST_TEXT_W,
    'best_val_uar':         round(BEST_VAL_UAR, 6),
    'best_val_accuracy':    round(BEST_VAL_ACC, 6),
    'best_val_macro_f1':    round(BEST_VAL_F1,  6),
    'test_accuracy':        round(test_acc, 6),
    'test_uar':             round(test_uar, 6),
    'test_macro_f1':        round(test_f1,  6),
    'hubert_test_uar':      round(hubert_test_uar, 6),
    'hubert_test_acc':      round(hubert_test_acc, 6),
    'hubert_test_macro_f1': round(hubert_test_f1,  6),
    'text_test_uar':        round(text_test_uar, 6),
    'text_test_acc':        round(text_test_acc, 6),
    'text_test_macro_f1':   round(text_test_f1,  6),
    'fusion_vs_hubert_uar': round(fusion_vs_hubert_uar, 6),
    'fusion_vs_text_uar':   round(fusion_vs_text_uar,   6),
    'fusion_vs_hubert_f1':  round(fusion_vs_hubert_f1,  6),
    'fusion_vs_text_f1':    round(fusion_vs_text_f1,    6),
    'weight_grid':          WEIGHT_GRID,
    'label_order':          EMOTIONS,
    'num_val_samples':      int(len(audio_val)),
    'num_test_samples':     int(len(audio_test)),
    'random_seed':          RANDOM_SEED,
}
with open(FUSION_DIR / 'fusion_final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

# Print manifest of all saved files
print(f'Output directory: {FUSION_DIR.resolve()}')
print()
expected = [
    'fusion_validation_results.csv',
    'fusion_test_predictions.csv',
    'fusion_final_metrics.json',
    'fusion_confusion_matrix.png',
    'fusion_weight_search.png',
    'text_val_predictions_with_probs.csv',
    'text_test_predictions_with_probs.csv',
]
for fname in expected:
    fpath  = FUSION_DIR / fname
    status = '[OK]    ' if fpath.exists() else '[MISSING]'
    size   = f'{fpath.stat().st_size:>10,} bytes' if fpath.exists() else ''
    print(f'  {status} {fname:<45} {size}')